## Metareligion Sales Analysis (*2023-2026*)
By *Wiraatmaja*

Metareligion is an archive-fashion resale business operating across Depop,
Grailed, Vinted and private sales. This notebook cleans and analyzes 111
sales I logged between Feb 2023 and Oct 2026, covering purchase cost, sale
price, fees, shipping, and realised profit per item.

Questions to answer:

1. Which channel actually protects margin once price tier is accounted
   for, and why?
2. Which categories and price tiers should Metareligion source more of,
   based on profit, margin and resale speed?

**Tools:** Python (pandas, matplotlib)

In [36]:
import pandas as pd

In [37]:
path = '/content/drive/MyDrive/Analytics Work/MG_Sales_Sheet1.csv'

In [38]:
# Cell 1 – mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
import os
os.listdir('/content/drive/MyDrive/Analytics Work')

['BWI Instagram & Ad Performance Data Analytics Report 22-25.docx',
 'BWI Instagram Content Analytics 25-26.docx',
 'MG_SALES_Sheet1.csv']

In [40]:
path = '/content/drive/MyDrive/Analytics Work/MG_SALES_Sheet1.csv'
df = pd.read_csv(path)

## 1. Data Overview

The raw Excel export has 111 sales logged between Feb 2023 and Oct 2026 (*2026 is partial - 10 months*)



In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Item                111 non-null    object 
 1   Brand               111 non-null    object 
 2   Category            111 non-null    object 
 3   Purchase Price (£)  111 non-null    int64  
 4   Sale Price (£)      111 non-null    float64
 5   Shipping (£)        111 non-null    int64  
 6   Platform Fee (£)    111 non-null    float64
 7   Profit (£)          111 non-null    float64
 8   Days to Sell        111 non-null    object 
 9   Date                111 non-null    object 
 10  Platform            111 non-null    object 
dtypes: float64(3), int64(2), object(6)
memory usage: 9.7+ KB


In [42]:
df['Days to Sell'].value_counts()

,count
Days to Sell,
10 Days,17
8 Days,16
5 Days,13
3 Days,10
12 Days,10
6 Days,8
14 Days,7
7 Days,7
30+ Days,4


In [43]:
df['Category'].value_counts()

,count
Category,
Jacket,42
Jeans,33
Accessories,10
T-Shirt,8
Long-Sleeve,6
T-shirt,5
Shoes,5
Bag,1
bag,1


In [44]:
df['Platform'].value_counts()

,count
Platform,
Depop,79
Grailed,17
Vinted,7
Private sale,4
Private Sale,4


### 1.1 Data issues log

Before cleaning anything, I ran `.info()`, `.value_counts()` and `.describe()` over the raw export to see if theres flaws that i made during logging in excel. A few things stood out:

- **`Days to Sell` is stored as text** (e.g. "5 Days"), so I can't average it or compare speed across platforms until it's converted to a number.

- Within that column, I logged "30+ Days" and "30 Days+", the same thing written two different ways, both mean I stopped tracking after 30 days, I'm treating both as *at least* 30 days and flagging them separately, rather than pretending I know the real number.

- **`Date`** is logged as month/year only, so it's also text right now, and it means this dataset is monthly-resolution, not daily. I can't look at day-of-week patterns, only month-to-month trends.

- **`Category`, `Platform`** and **`Brand`** all have case-inconsistent duplicates I made, e.g. "T-Shirt" vs "T-shirt", "Private Sale" vs "Private sale", "GINZI" vs "Ginzi", which silently splits one category into two until I standardise the text.

- **`Brand`** has both **"No Brand"** and **"Unknown"** logged separately. To me these mean the same thing (*I don't have a designer name for the item*), so I'm merging them.

- On the positive side: there are no missing values anywhere in the 111 rows, and the `Profit (£)` column checks out exactly against Sale / Purchase / Shipping / Fee for every row, so I'm confident the core numbers weren't corrupted.

In [45]:
# Standardize case
df['Category'] = df['Category'].str.strip().str.title()
df['Platform'] = df['Platform'].str.strip().str.title()

In [46]:
df['Brand'].value_counts()

,count
Brand,
Saint Laurent Paris,12
Hysteric Glamour,11
Rick Owens,9
L.G.B,8
Dior Homme,4
EKAM,4
Balmain,4
5351 Pour les Hommes,3
Unknown,3


In [47]:
df['Category'] = df['Category'].str.strip().str.title()
df['Platform'] = df['Platform'].str.strip().str.title()
df['Brand'] = df['Brand'].str.strip().str.title()

In [48]:
df['Brand'] = df['Brand'].replace('No Brand', 'Unknown')

In [49]:
df['Date'] = pd.to_datetime(df['Date'], format='%m/%Y')

In [50]:
df['days_censored'] = df['Days to Sell'].str.contains(r'\+')

In [51]:
df['days_to_sell'] = df['Days to Sell'].str.extract(r'(\d+)').astype(int)

In [52]:
df.info()
df.head(10)
df[df['days_censored']][['Item', 'Days to Sell', 'days_to_sell', 'days_censored']]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Item                111 non-null    object        
 1   Brand               111 non-null    object        
 2   Category            111 non-null    object        
 3   Purchase Price (£)  111 non-null    int64         
 4   Sale Price (£)      111 non-null    float64       
 5   Shipping (£)        111 non-null    int64         
 6   Platform Fee (£)    111 non-null    float64       
 7   Profit (£)          111 non-null    float64       
 8   Days to Sell        111 non-null    object        
 9   Date                111 non-null    datetime64[ns]
 10  Platform            111 non-null    object        
 11  days_censored       111 non-null    bool          
 12  days_to_sell        111 non-null    int64         
dtypes: bool(1), datetime64[ns](1), float64(3), int64(3

,Item,Days to Sell,days_to_sell,days_censored
59,Napoleon Gilet,30+ Days,30,True
82,Cross-zip military jk,30+ Days,30,True
90,Shotgun T,30+ Days,30,True
94,Longsleeve,30+ Days,30,True
107,YSL Logo T Shirt,30 Days+,30,True


In [53]:
df['Category'].value_counts()
df['Platform'].value_counts()
df['Brand'].value_counts()

,count
Brand,
Saint Laurent Paris,12
Hysteric Glamour,11
Rick Owens,9
L.G.B,8
Unknown,4
Dior Homme,4
Ekam,4
Balmain,4
Roen,3


Datas all cleaned up. I've merged the case-duplicate categories into one (Brand's "Unknown" count went from 3 to 4 after folding in "No Brand", and "Ginzi"/"GINZI" merged into a single count of 2). Now I can build the actual profitability metrics.

## 2. Unit Economics

Raw profit in £ is a bit misleading on its own, £300 profit on a £3,000 jacket is a worse outcome than £50 profit on a £100 t-shirt. So before comparing platforms or categories, I'm building three ratio metrics:

- **Margin %**, profit as a share of sale price. Tells me how much of what a buyer pays actually turns into profit.
- **ROI %**, profit as a share of what I originally paid. Tells me how hard my capital is working, which is the number that should actually guide what I buy more of.
- **Cost drag %**, shipping + platform fees as a share of sale price. Tells me how much of the sale is eaten before I even get to profit.

In [54]:
df['margin_pct'] = (df['Profit (£)'] / df['Sale Price (£)'] * 100).round(1)
df['roi_pct'] = (df['Profit (£)'] / df['Purchase Price (£)'] * 100).round(1)
df['cost_drag_pct'] = ((df['Shipping (£)'] + df['Platform Fee (£)']) / df['Sale Price (£)'] * 100).round(1)

df[['Item', 'Purchase Price (£)', 'Sale Price (£)', 'Profit (£)', 'margin_pct', 'roi_pct', 'cost_drag_pct']].sort_values('roi_pct', ascending=False).head(10)

,Item,Purchase Price (£),Sale Price (£),Profit (£),margin_pct,roi_pct,cost_drag_pct
54,SLP Glasses,0,75.0,62.5,83.3,inf,16.7
90,Shotgun T,7,100.0,68.0,68.0,971.4,25.0
99,Biker Jeans,13,144.0,79.0,54.9,607.7,36.1
35,L leather patch jeans,15,108.0,77.2,71.5,514.7,14.6
30,Slave Denim jk,40,250.0,180.0,72.0,450.0,12.0
100,2015 Crash Denim,35,250.0,149.5,59.8,427.1,26.2
88,RL American Flag Hoodie,25,125.0,100.0,80.0,400.0,0.0
83,Plaid Hooded shirt,15,75.0,55.0,73.3,366.7,6.7
106,Raw Biker Jeans,40,236.0,135.0,57.2,337.5,25.8
31,Studded Jeans,40,175.0,130.0,74.3,325.0,2.9


In [55]:
# ROI is undefined when Purchase Price is 0 — replace inf with NaN so it's excluded from averages automatically
df['roi_pct'] = df['roi_pct'].replace([float('inf'), -float('inf')], pd.NA)

One item (SLP Glasses) was a gift, so its Purchase Price i logged as £0, which breaks ROI with a divide-by-zero and returns `inf`. Since ROI is genuinely undefined when there's no capital at risk, I'm excluding it from ROI averages rather than letting `inf` quietly wreck every average it touches.

In [56]:
platform_summary = df.groupby('Platform').agg(
    n_sales=('Item', 'count'),
    total_profit=('Profit (£)', 'sum'),
    avg_sale_price=('Sale Price (£)', 'mean'),
    avg_margin_pct=('margin_pct', 'mean'),
    avg_roi_pct=('roi_pct', 'mean'),
    avg_days_to_sell=('days_to_sell', 'mean'),
    avg_cost_drag_pct=('cost_drag_pct', 'mean')
).round(1).sort_values('avg_margin_pct', ascending=False)

platform_summary

,n_sales,total_profit,avg_sale_price,avg_margin_pct,avg_roi_pct,avg_days_to_sell,avg_cost_drag_pct
Platform,,,,,,,
Vinted,7,525.0,119.3,58.6,190.571429,12.0,4.6
Private Sale,8,464.0,104.6,54.3,299.3875,11.1,21.4
Grailed,17,2444.5,294.2,48.0,227.258824,8.3,26.2
Depop,79,4516.1,125.9,43.2,145.770513,10.6,15.7


2.1 Platform

 Depop earns the most total profit just from volume (79 sales), but has the lowest margin and takes the longest to sell. Grailed and Vinted look better on margin, but their sample sizes are small (7-8 sales), and Grailed also lists pricier items on average (£294 vs £126), which could explain the margin gap on its own. From running the business across all three platforms, I've noticed expensive, rare pieces consistently do better on Grailed. The market audience pays for scarcity, while the same item listed cheaper on Depop often doesnt sell. That matches the data: Grailed's margin holds up even with higher fees, which only works if buyers there are paying a dime.

In [57]:
# Bucket into price tiers to compare like-for-like
df['price_tier'] = pd.cut(df['Sale Price (£)'], bins=[0, 100, 200, 1000], labels=['Under £100', '£100-200', 'Over £200'])

pd.crosstab(df['Platform'], df['price_tier'])

price_tier,Under £100,£100-200,Over £200
Platform,,,
Depop,34,32,13
Grailed,3,2,12
Private Sale,7,0,1
Vinted,3,3,1


That's confirmed below: 71% of my Grailed sales (12 of 17) are over £200, versus only 16% of my Depop sales (13 of 79). I'm not comparing two platforms doing the same job. I'm comparing "where I list my cheap stock" against "where I list my best pieces." Any margin difference I saw above could just be a price-tier artifact.

In [58]:
df.groupby(['price_tier', 'Platform']).agg(
    n=('Item', 'count'),
    avg_margin_pct=('margin_pct', 'mean'),
    avg_cost_drag_pct=('cost_drag_pct', 'mean')
).round(1)


/tmp/ipykernel_2153/422566693.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['price_tier', 'Platform']).agg(


n  avg_margin_pct  avg_cost_drag_pct
price_tier Platform                                           
Under £100 Depop         34            38.9               17.8
           Grailed        3            40.1               34.6
           Private Sale   7            54.4               22.5
           Vinted         3            53.0                0.0
£100-200   Depop         32            45.9               14.5
           Grailed        2            46.6               36.9
           Private Sale   0             NaN                NaN
           Vinted         3            60.1                8.6
Over £200  Depop         13            47.7               12.9
           Grailed       12            50.3               22.3
           Private Sale   1            53.3               13.3
           Vinted         1            70.8                6.2

**2.2 key test**

Once I compare margin % within the same price band, the gap between Depop and Grailed almost disappears, at most ~1-3 percentage points in every tier, nowhere near the ~5-point gap in the raw numbers. So Grailed isn't intrinsically a "better margin" platform.

What *does* hold up in every tier is cost drag: Grailed's fees + shipping eat roughly double the percentage of revenue that Depop's do (e.g. 34.6% vs 17.8% under £100). Grailed is the more expensive platform to sell on. The fact that margins still come out roughly even means Grailed buyers are paying enough of a premium over what I'd get on Depop to reciprocate that extra cost. In other words, it's not the platform's fee structure that protects margin, it's the audience's willingness to pay for archive pieces that happens to offset Grailed's worse fees.

In [59]:
df.groupby('Category').agg(
    n=('Item','count'), total_profit=('Profit (£)','sum'),
    avg_margin_pct=('margin_pct','mean'), avg_days=('days_to_sell','mean')
).round(1).sort_values('total_profit', ascending=False)

,n,total_profit,avg_margin_pct,avg_days
Category,,,,
Jacket,42,3155.2,46.6,9.5
Jeans,33,3148.7,47.7,11.2
Shoes,5,410.0,35.1,6.8
Accessories,10,397.9,42.7,10.3
T-Shirt,13,379.3,44.1,13.2
Long-Sleeve,6,280.5,39.1,10.5
Bag,2,178.0,65.8,6.5


2.3 **By category:**

Jacket and Jeans are clearly the backbone of the business, 75 of my 111 sales (68%) and roughly 86% of total profit, with solid margins (46-48%) and reasonably quick turnover (9-11 days). Any sourcing decision should start from keeping this pipeline full.

Bag shows the best margin by far (65.8%) and Shoes sells the fastest (6.8 days), but both only have n=2 and n=5 respectively, too small to generalize from. I'm noting these as things to watch, not conclusions to act on yet.

In [60]:
df['year'] = df['Date'].dt.year

yearly = df.groupby('year').agg(
    n=('Item','count'), total_profit=('Profit (£)','sum'), avg_margin_pct=('margin_pct','mean')
).round(1)

platform_mix = pd.crosstab(df['year'], df['Platform'], normalize='index').round(2)

yearly
platform_mix

Platform,Depop,Grailed,Private Sale,Vinted
year,,,,
2023,1.00,0.00,0.00,0.00
2024,1.00,0.00,0.00,0.00
2025,0.76,0.05,0.08,0.11
2026,0.26,0.48,0.16,0.10


In [61]:
platform_mix = (pd.crosstab(df['year'], df['Platform'], normalize='index') * 100).round(0)
platform_mix

Platform,Depop,Grailed,Private Sale,Vinted
year,,,,
2023,100.0,0.0,0.0,0.0
2024,100.0,0.0,0.0,0.0
2025,76.0,5.0,8.0,11.0
2026,26.0,48.0,16.0,10.0


## 3. Trend Over Time

Laid out by year, this is the business's channel shift happening in front of me: Depop was 100% of sales in 2023-2024, declined through 2025, and by 2026 Grailed had overtaken it (48% vs 26% share). Profit and margin both hit their best levels yet in 2026 (£3,281.5 profit, 51.4% average margin), right as that shift happened. (Worth flagging: 2026 only covers 10 months in this dataset, so it's not a fair full-year comparison against 2023-2025 yet.)

**Context the numbers alone can't show:** the move away from Depop wasn't really a clean A/B test where I picked the better platform. Depop sales for me have genuinely dried up over the past year or so, I haven't seen the same traffic I used to. Vinted, meanwhile, only works while I'm physically in the UK, so it isn't a channel I can lean into from Indonesia. That makes Grailed more of a "what's actually still available to me" outcome than a deliberate platform preference. Combined with the margin finding above, Grailed's audience absorbs its higher fees through willingness to pay more, the picture is: Depop became less viable, Vinted isn't accessible from where I am, and Grailed happened to be both available and, for the pieces I put there, workable on margin.

## 4. Recommendations

1. **Keep Jacket and Jeans as the core of sourcing.** They're 68% of sales and ~86% of profit, with good margins and fast turnover, the most proven part of the business.
2. **Lean further into Grailed for higher-value pieces**, not because its fee structure is better (it isn't, fees run roughly double Depop's at every price point), but because its audience's willingness to pay absorbs that cost. This only works for items that can carry a premium price; it's not automatically the right channel for lower-value stock.
3. **Treat Bag and Shoes as hypotheses, not strategy.** Their numbers look good but the sample sizes (n=2, n=5) are too small to build a sourcing decision on. Worth tracking as more sales come in.
4. **Watch how 2026 closes out** before treating this year's margin jump as the new normal, right now it's a partial year being compared against full years.

## 5. Limitations

- Vinted (n=7), Private Sale (n=8), Bag (n=2) and Shoes (n=5) all have small samples: their averages can move a lot from one unusual sale.
- 2026 data only runs through October, so it isn't directly comparable to the full years before it.
- Dates are logged at month resolution, so I can't look at anything finer (day-of-week, exact selling window).
- Platform and time are tangled together in this data, the Depop -> Grailed shift happened over the same period the business matured generally, so I can't fully separate "Grailed is the reason margin improved" from "the business got better at pricing/sourcing over time anyway."